# Day 93 — Convolution

This notebook covers 2D Convolution (`nn.Conv2d`), Local Connectivity & Spatial Weight Sharing, Kernel Size, Stride & Padding Mechanics, Spatial Dimension Formula ($\lfloor \frac{H_{\text{in}} + 2P - K}{S} \rfloor + 1$), and Parameter Efficiency Analysis

## Validation goals
- Implement `expected_spatial_dim` to calculate output height and width across varying kernel sizes, padding, and strides
- Contrast same-resolution convolutions (`kernel_size=3, padding=1, stride=1`) against spatial downsampling convolutions (`stride=2`) on `(8, 3, 32, 32)` RGB batches
- Audit `nn.Conv2d` parameter counts (`448` parameters for `3 -> 16` channels with a $3 \times 3$ kernel) and contrast spatial weight sharing against dense `nn.Linear` layers
- Analyze translation equivariance, border pixel centering under zero-padding, and spatial shrinkage when `padding=0

### 1. How a 2D Convolution Works

A dense (`nn.Linear`) layer connects every input pixel to every hidden unit—ignoring spatial locality and exploding parameter counts on large images. A convolution layer instead slides a small learnable kernel (or filter), such as a $3 \times 3$ weight grid, across the input image:
- At each spatial position, the kernel multiplies overlapping pixel values element-wise and sums them (plus a bias term) to produce one value in the output feature map.
- Because the same kernel weights are reused across every spatial location (parameter sharing), a filter that detects a vertical edge in the top-left corner can detect a vertical edge anywhere in the image.
- Adding multiple output channels (out_channels) lets a single `nn.Conv2d` layer learn multiple distinct detectors in parallel.

### 2. Padding, Stride, and Output Spatial Dimensions

Three hyperparameters shape how a Conv2d layer moves across an image and what output height/width it produces:
- Kernel Size (kernel_size): Height and width of the sliding window (e.g., 3 for $3 \times 3$).
- Stride (stride): How many pixels the kernel steps each move (stride=1 visits positions pixel-by-pixel; stride=2 skips every other pixel, halving spatial resolution).
- Padding (padding): Zero-valued border pixels added around the input grid. Adding padding=1 with a $3 \times 3$ kernel and stride=1 preserves the exact input height and width while letting border pixels participate in full windows.

For input spatial dimension $H_{\text{in}}$, kernel size $K$, padding $P$, and stride $S$, the output height is:$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2P - K}{S} \right\rfloor + 1$$The same formula applies to width ($W_{\text{out}}$).

### 3. Implementing & Inspecting nn.Conv2d

Let's write a script that runs a batch of NCHW image tensors through `nn.Conv2d`, verifies output shapes against the spatial formula, and prints parameter counts.

In [1]:
import torch
import torch.nn as nn


def expected_spatial_dim(size_in: int, kernel_size: int, padding: int, stride: int) -> int:
    """Computes output height or width for a standard 2D convolution."""
    return ((size_in + 2 * padding - kernel_size) // stride) + 1


if __name__ == "__main__":
    torch.manual_seed(42)

    # Batch of 8 RGB images: (N=8, C=3, H=32, W=32)
    x = torch.randn((8, 3, 32, 32), dtype=torch.float32)

    # Conv layer: 3 input channels -> 16 feature maps, 3x3 kernel, padding=1, stride=1
    conv_same = nn.Conv2d(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        stride=1,
        padding=1,
    )

    # Strided conv layer: 16 -> 32 feature maps, 3x3 kernel, padding=1, stride=2
    conv_stride2 = nn.Conv2d(
        in_channels=16,
        out_channels=32,
        kernel_size=3,
        stride=2,
        padding=1,
    )

    h1 = x.shape[2]
    w1 = x.shape[3]
    out1 = conv_same(x)
    exp_h1 = expected_spatial_dim(h1, kernel_size=3, padding=1, stride=1)
    exp_w1 = expected_spatial_dim(w1, kernel_size=3, padding=1, stride=1)

    out2 = conv_stride2(out1)
    exp_h2 = expected_spatial_dim(out1.shape[2], kernel_size=3, padding=1, stride=2)
    exp_w2 = expected_spatial_dim(out1.shape[3], kernel_size=3, padding=1, stride=2)

    conv1_params = sum(p.numel() for p in conv_same.parameters())

    print("--- Day 93: 2D Convolution & Output Shapes ---")
    print(f"Input shape:       {tuple(x.shape)}")
    print(f"After conv_same:   {tuple(out1.shape)} (expected H={exp_h1}, W={exp_w1})")
    print(f"After conv_stride2:{tuple(out2.shape)} (expected H={exp_h2}, W={exp_w2})")
    print(f"conv_same params:  {conv1_params} (3*3*3*16 weights + 16 biases)")

--- Day 93: 2D Convolution & Output Shapes ---
Input shape:       (8, 3, 32, 32)
After conv_same:   (8, 16, 32, 32) (expected H=32, W=32)
After conv_stride2:(8, 32, 16, 16) (expected H=16, W=16)
conv_same params:  448 (3*3*3*16 weights + 16 biases)


### 4. Key Takeaways for CNN Design

- Channel Count Controls Feature Capacity: Increasing out_channels from 16 to 32 doubles the number of distinct feature maps produced at each spatial location, without changing the input image's height or width when padding=1 and stride=1.
- Weight Sharing Keeps Parameter Counts Manageable: A `3x3` conv mapping `3 -> 16 `channels uses only 448 parameters (`3 * 3 * 3 * 16 + 16`), independent of whether the input image is `32x32` or ``224x224``.
- Spatial Downsampling Happens via Stride or Pooling: Using `stride=2` reduces spatial dimensions by roughly half, trading resolution for larger effective receptive fields in deeper layers.

### Guided Question:

Why does weight sharing across spatial positions in `nn.Conv2d` make `CNNs` far more parameter-efficient and translation-friendly than a fully connected layer on raw image pixels?

### Answer:

In nn.Conv2d, weight sharing ties two mechanisms together: local connectivity (each output unit reads only a small spatial patch, or receptive field, rather than the entire image) and shared weights (the same kernel weights are applied across every spatial position (`h, w`)).

**Why Parameter Efficiency Explodes in Fully Connected Layers**

A `nn.Linear` layer flattens a tensor of shape (C_in, H, W) into a 1D vector of length C_in * H * W. For C_out output features, a `nn.Linear` layer requires:$$\text{Params}_{\text{Linear}} = (C_{\text{in}} \cdot H \cdot W) \cdot C_{\text{out}} + C_{\text{out}}$$By contrast, `nn.Conv2d(C_in, C_out, kernel_size=(K_h, K_w)`) uses a kernel tensor of shape (C_out, C_in, K_h, K_w) (plus optional C_out biases), independent of H and W:$$\text{Params}_{\text{Conv2d}} = C_{\text{out}} \cdot C_{\text{in}} \cdot K_h \cdot K_w + C_{\text{out}}$$For example, consider a 128 x 128 RGB image (C_in = 3, 16,384 pixels, or 49,152 input values) mapped to a 128 x 128 feature map with 32 channels (C_out = 32, 524,288 output activations):
- `nn.Linear` (ignoring bias): 49,152 * 524,288 = 25,769,803,776 weights (over 25.7 billion parameters).
- `nn.Conv2d` with 3 x 3 kernel (ignoring bias): 32 * 3 * 3 * 3 = 864 weights.

That is a reduction of roughly 30 million times in parameters for that layer. Because `nn.Conv2d` parameter count does not scale with H * W, the same weights can process 128 x 128 or 512 x 512 inputs without inflating the weight matrix, reducing overfitting and gradient noise.

**Why Weight Sharing Makes CNNs Translation-Friendly**

In natural images, visual statistics are stationary across space: a horizontal edge, corner, or texture is just as meaningful at pixel (20, 45) as at (180, 310).
- Without weight sharing (nn.Linear): Every pixel-to-output connection has an independent weight. If a pattern shifts a few pixels to the left or right, the activations move to different input coordinates that have never seen that pattern during training unless the training set covers every position densely.
- With weight sharing (nn.Conv2d): One learned detector slides across the entire grid. If an edge detector fires strongly at (20, 45) for an input X, shifting X by (dh, dw) shifts that feature map response by (dh, dw) (assuming boundary padding behavior is consistent).

### Guided Question:

For a `64x64` feature map passed through `nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1, padding=0)`, what are the resulting output height and width?

### Answer:

The resulting output height and width are 62 x 62 (giving a tensor of shape (batch_size, 16, 62, 62)).

For standard nn.Conv2d with default dilation=1, the spatial output size along each dimension is:$$\text{Out} = \left\lfloor \frac{\text{In} + 2 \cdot \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1$$Substituting $\text{In} = 64$, $\text{padding} = 0$, $\text{kernel\_size} = 3$, and $\text{stride} = 1$:$$\text{Out} = \left\lfloor \frac{64 + 2(0) - 3}{1} \right\rfloor + 1 = 61 + 1 = 62$$

Because there is no padding (`padding=0`), a 3 x 3 kernel cannot center on the outermost border pixels without stepping outside the feature map, reducing each spatial dimension by kernel_size - 1 = 2 pixels.

### Guided Question:

Why does `padding=1` on a `3x3 (stride=1)` convolution keep spatial dimensions unchanged and allow edge pixels to be centered in kernel windows?

### Answer:

A `3 x 3` kernel has odd dimensions, so it has an exact center element at offset `(-1, -1) to (+1, +1)` relative to its center. Adding padding=1 appends a 1-pixel border around all four sides of the input feature map, expanding a spatial size of `N to N + 2`.

**Centering Edge and Corner Pixels**

- Edge pixels: Without padding, the center of a `3 x 3` window cannot sit on index 0 or N - 1 because one neighbor row or column does not exist. Adding a 1-pixel border around the map supplies that missing neighbor row or column so index 0 and index N - 1 can sit directly under the kernel's center weight (1, 1).
- Corner pixels: Adding 1 layer on both the top/bottom and left/right creates corner cells in the padded border, allowing corner pixels like (0, 0) to be centered inside a full `3 x 3` neighborhood as well (with the out-of-bounds slots filled by zeros in standard zero-padding).

**Preserving Spatial Dimensions**

Using the standard convolution output formula (with dilation=1):$$\text{Out} = \left\lfloor \frac{\text{In} + 2 \cdot \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1$$Plug in kernel_size = 3, stride = 1, and padding = 1 for an input dimension H:$$\text{Out} = \left\lfloor \frac{H + 2(1) - 3}{1} \right\rfloor + 1 = (H - 1) + 1 = H$$

The same holds for width W. Because 2 * padding (+2) exactly cancels kernel_size - 1 (-2) when stride = 1, the output feature map keeps the exact (H, W) dimensions of the input.

**Practical Consequences**

- Deeper stacks without shrinking: You can stack many `3 x 3, stride=1, padding=1 `layers and easily predict tensor shapes across the network (and align skip connections or residual branches without resizing).
- Border information retention: Edge features stay represented in the spatial grid instead of being progressively trimmed away layer after layer.
- Zero-padding trade-off: Standard `nn.Conv2d` pads with 0s by default, which introduces artificial zero values at the border that can slightly bias activations near image edges.